In [1]:
# ============================================
# Colab-ready: Diagnostic Results (channels + geometric calibration protocols)
# Generates in ./results:
#   - fig_dephasing.jpg
#   - fig_depolarizing.jpg
#   - fig_amplitude_damping.jpg
#   - panel_channels.jpg
#   - fig_equalF_GDI_GAC.jpg
#   - fig_equalLB_efficiency.jpg
#   - table_protocols_summary.csv
#   - table_equalF_uncertainty.csv
# ============================================

import os, shutil
import numpy as np
import pandas as pd
from numpy.random import default_rng
from scipy.linalg import sqrtm
import matplotlib.pyplot as plt

# -----------------------------
# Output folder
# -----------------------------
RESULTS_DIR = "./results"
if os.path.exists(RESULTS_DIR):
    shutil.rmtree(RESULTS_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)

# -----------------------------
# Configuration
# -----------------------------
N_POINTS = 61
DEPTH = 32
N_RANDOM = 24
rng = default_rng(42)

RANGE_DEPH = (0.005, 0.03)
RANGE_DEPO = (0.015, 0.075)
RANGE_AD = (0.80, 1.00)

EPS = 1e-10

# -----------------------------
# Linear algebra + Pauli matrices
# -----------------------------
I2 = np.eye(2, dtype=np.complex128)
X = np.array([[0, 1], [1, 0]], dtype=np.complex128)
Y = np.array([[0, -1j], [1j, 0]], dtype=np.complex128)
Z = np.array([[1, 0], [0, -1]], dtype=np.complex128)

# -----------------------------
# State utilities
# -----------------------------
def pure_to_rho(psi):
    psi = np.asarray(psi, dtype=np.complex128).reshape(2, 1)
    psi = psi / np.linalg.norm(psi)
    return psi @ psi.conj().T


def haar_random_pure_qubit(n):
    states = []
    for _ in range(n):
        z = rng.normal(size=2) + 1j * rng.normal(size=2)
        psi = z / np.linalg.norm(z)
        states.append(pure_to_rho(psi))
    return states


def sample_states(n_random):
    cardinal = [
        pure_to_rho([1, 0]),
        pure_to_rho([0, 1]),
        pure_to_rho([1 / np.sqrt(2), 1 / np.sqrt(2)]),
        pure_to_rho([1 / np.sqrt(2), -1 / np.sqrt(2)]),
        pure_to_rho([1 / np.sqrt(2), 1j / np.sqrt(2)]),
        pure_to_rho([1 / np.sqrt(2), -1j / np.sqrt(2)]),
    ]
    return cardinal + haar_random_pure_qubit(n_random)


# Fixed ensemble for all computations
STATES = sample_states(N_RANDOM)

# -----------------------------
# Fidelity and Bures distance
# -----------------------------
def fidelity(rho, sigma):
    sr = sqrtm(rho)
    inner = sr @ sigma @ sr
    inner = (inner + inner.conj().T) / 2.0
    root = sqrtm(inner)
    val = np.real(np.trace(root)) ** 2
    return float(np.clip(val, 0.0, 1.0))


def bures_distance(rho, sigma):
    F = fidelity(rho, sigma)
    return float(np.sqrt(max(0.0, 2.0 * (1.0 - np.sqrt(F)))))


# -----------------------------
# Channels
# -----------------------------
def apply_kraus(rho, Ks):
    out = np.zeros((2, 2), dtype=np.complex128)
    for K in Ks:
        out += K @ rho @ K.conj().T
    out = (out + out.conj().T) / 2.0
    tr = np.real(np.trace(out))
    return out / tr if tr != 0 else I2 / 2


def dephasing_kraus(p):
    return [np.sqrt(1 - p) * I2, np.sqrt(p) * Z]


def depolarizing_kraus(p):
    a0 = np.sqrt(1 - 3 * p / 4)
    a = np.sqrt(p / 4)
    return [a0 * I2, a * X, a * Y, a * Z]


def amplitude_damping_kraus(g):
    g = min(max(float(g), 0.0), 1.0)
    K0 = np.array([[1, 0], [0, np.sqrt(1 - g)]], dtype=np.complex128)
    K1 = np.array([[0, np.sqrt(g)], [0, 0]], dtype=np.complex128)
    return [K0, K1]


# -----------------------------
# Trajectories and metrics
# -----------------------------
def trajectory(rho0, kraus_fn, param, depth):
    traj = [rho0]
    Ks = kraus_fn(param)
    rho = rho0
    for _ in range(depth):
        rho = apply_kraus(rho, Ks)
        traj.append(rho)
    return traj


def trajectory_length_bures(traj):
    return sum(bures_distance(traj[i], traj[i + 1]) for i in range(len(traj) - 1))


def gdi(traj, eps=EPS):
    """
    GDI = L_B / D_B.

    Fixed-point trajectories have L_B = 0 and D_B = 0.
    In that degenerate case, GDI is undefined and is returned as NaN.
    """
    LB = trajectory_length_bures(traj)
    DB = bures_distance(traj[0], traj[-1])

    if LB < eps and DB < eps:
        return np.nan

    if DB < eps:
        return np.nan

    return float(LB / DB)


def gac_mean(traj, eps=EPS):
    """
    Mean local alignment using a Bures-distance law-of-cosines surrogate.

    Fixed or degenerate trajectories are returned as NaN.
    """
    T = traj[-1]
    vals = []

    for i in range(len(traj) - 1):
        A = traj[i]
        B = traj[i + 1]

        dAB = bures_distance(A, B)
        dAT = bures_distance(A, T)
        dBT = bures_distance(B, T)

        denom = 2.0 * dAB * dAT

        if denom <= eps:
            continue

        cos_th = (dAB**2 + dAT**2 - dBT**2) / denom
        vals.append(np.clip(np.real(cos_th), -1.0, 1.0))

    return float(np.mean(vals)) if vals else np.nan


def bures_speeds_and_accel(traj):
    v = [bures_distance(traj[i], traj[i + 1]) for i in range(len(traj) - 1)]
    a = []

    for i in range(1, len(v) - 1):
        a.append(abs(v[i + 1] - 2 * v[i] + v[i - 1]))

    max_a = max(a) if a else 0.0
    return v, a, max_a


def safe_nanmean(x):
    x = np.asarray(x, dtype=float)
    return float(np.nanmean(x)) if np.any(~np.isnan(x)) else np.nan


def safe_nanstd(x):
    x = np.asarray(x, dtype=float)
    valid = x[~np.isnan(x)]
    return float(np.std(valid, ddof=1)) if len(valid) > 1 else np.nan


# -----------------------------
# Mean metrics for fixed parameter
# -----------------------------
def mean_metrics_for_param(kraus_fn, param, depth, states):
    Fs, DBs, GDIs, GACs, LBs, maxAs = [], [], [], [], [], []
    Ks = kraus_fn(float(param))

    for rho0 in states:
        traj = [rho0]
        rho = rho0

        for _ in range(depth):
            rho = apply_kraus(rho, Ks)
            traj.append(rho)

        Fs.append(fidelity(traj[0], traj[-1]))
        DBs.append(bures_distance(traj[0], traj[-1]))
        GDIs.append(gdi(traj))
        GACs.append(gac_mean(traj))
        LBs.append(trajectory_length_bures(traj))

        _, _, max_a = bures_speeds_and_accel(traj)
        maxAs.append(max_a)

    return {
        "F": float(np.mean(Fs)),
        "F_std": float(np.std(Fs, ddof=1)),
        "DB": float(np.mean(DBs)),
        "DB_std": float(np.std(DBs, ddof=1)),
        "GDI": safe_nanmean(GDIs),
        "GDI_std": safe_nanstd(GDIs),
        "GAC": safe_nanmean(GACs),
        "GAC_std": safe_nanstd(GACs),
        "LB": float(np.mean(LBs)),
        "LB_std": float(np.std(LBs, ddof=1)),
        "max_aB": float(np.mean(maxAs)),
        "max_aB_std": float(np.std(maxAs, ddof=1)),
        "n_valid_GDI": int(np.sum(~np.isnan(GDIs))),
        "n_valid_GAC": int(np.sum(~np.isnan(GACs))),
        "n_total": len(states),
    }


# -----------------------------
# Parameter sweep
# -----------------------------
def run_sweep_means(channel_name, kraus_fn, grid, depth, states):
    rows = []

    for param in grid:
        metrics = mean_metrics_for_param(kraus_fn, float(param), depth, states)
        rows.append({
            "channel": channel_name,
            "param": float(param),
            **metrics,
        })

    return pd.DataFrame(rows)


grid_deph = np.linspace(*RANGE_DEPH, N_POINTS)
grid_depo = np.linspace(*RANGE_DEPO, N_POINTS)
grid_ad = np.linspace(*RANGE_AD, N_POINTS)

df_deph = run_sweep_means("dephasing", dephasing_kraus, grid_deph, DEPTH, STATES)
df_depo = run_sweep_means("depolarizing", depolarizing_kraus, grid_depo, DEPTH, STATES)
df_ad = run_sweep_means("amplitude_damping", amplitude_damping_kraus, grid_ad, DEPTH, STATES)

# -----------------------------
# Plot channel curves
# -----------------------------
def plot_channel_curves(df, title, xlabel, outpath):
    d = df.sort_values("param")

    plt.figure(figsize=(6, 4))
    plt.plot(d["param"], d["F"], label="Fidelity")
    plt.plot(d["param"], d["DB"], label="Bures distance")
    plt.plot(d["param"], d["GDI"], label="GDI")
    plt.plot(d["param"], d["GAC"], label="GAC")

    plt.xlabel(xlabel)
    plt.ylabel("Index value")
    plt.title(title)
    plt.ylim(0, 1.2)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=300, format="jpg")
    plt.close()


plot_channel_curves(df_deph, "Dephasing channel", "Noise parameter p", f"{RESULTS_DIR}/fig_dephasing.jpg")
plot_channel_curves(df_depo, "Depolarizing channel", "Noise parameter p", f"{RESULTS_DIR}/fig_depolarizing.jpg")
plot_channel_curves(df_ad, "Amplitude damping channel", "Damping parameter γ", f"{RESULTS_DIR}/fig_amplitude_damping.jpg")

# -----------------------------
# Panel with 3 subplots
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)

for ax, df, title, xlabel in [
    (axes[0], df_deph, "Dephasing", "Noise parameter p"),
    (axes[1], df_depo, "Depolarizing", "Noise parameter p"),
    (axes[2], df_ad, "Amplitude damping", "Damping parameter γ"),
]:
    d = df.sort_values("param")
    ax.plot(d["param"], d["F"], label="Fidelity")
    ax.plot(d["param"], d["DB"], label="Bures distance")
    ax.plot(d["param"], d["GDI"], label="GDI")
    ax.plot(d["param"], d["GAC"], label="GAC")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylim(0, 1.2)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Index value")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.subplots_adjust(bottom=0.18)
plt.savefig(f"{RESULTS_DIR}/panel_channels.jpg", dpi=300, format="jpg")
plt.close()

print("Channel figures saved in ./results")

# -----------------------------
# Helpers to find target parameters
# -----------------------------
def find_param_for_target(kraus_fn, depth, states, target, lo, hi, metric_key="F", tol=1e-4, maxit=60):
    def mval(x):
        return mean_metrics_for_param(kraus_fn, x, depth, states)[metric_key]

    f_lo = mval(lo)
    f_hi = mval(hi)

    if not ((f_lo >= target >= f_hi) or (f_lo <= target <= f_hi)):
        raise ValueError(
            f"Target {target:.5f} not bracketed in [{lo:.5f}, {hi:.5f}] "
            f"for metric {metric_key}: f(lo)={f_lo:.5f}, f(hi)={f_hi:.5f}"
        )

    a, b = lo, hi

    for _ in range(maxit):
        mid = 0.5 * (a + b)
        f_mid = mval(mid)

        if abs(f_mid - target) < tol:
            return float(mid), float(f_mid)

        if f_lo < f_hi:
            if f_mid < target:
                a = mid
            else:
                b = mid
        else:
            if f_mid > target:
                a = mid
            else:
                b = mid

    return float(mid), float(f_mid)


def find_param_or_nearest(kraus_fn, depth, states, target, grid, metric_key="F"):
    lo, hi = float(grid[0]), float(grid[-1])

    try:
        return find_param_for_target(kraus_fn, depth, states, target, lo, hi, metric_key=metric_key)
    except Exception:
        rows = []
        for p in grid:
            val = mean_metrics_for_param(kraus_fn, float(p), depth, states)[metric_key]
            rows.append((float(p), float(val)))

        p_star, val_star = min(rows, key=lambda t: abs(t[1] - target))
        return float(p_star), float(val_star)


# -----------------------------
# Protocol P2: Equal final fidelity
# -----------------------------
ref_p = float(np.median(grid_deph))
ref_metrics = mean_metrics_for_param(dephasing_kraus, ref_p, DEPTH, STATES)
F_star = ref_metrics["F"]

p_deph_P2, _ = find_param_or_nearest(dephasing_kraus, DEPTH, STATES, F_star, grid_deph, metric_key="F")
p_depo_P2, _ = find_param_or_nearest(depolarizing_kraus, DEPTH, STATES, F_star, grid_depo, metric_key="F")
g_ad_P2, _ = find_param_or_nearest(amplitude_damping_kraus, DEPTH, STATES, F_star, grid_ad, metric_key="F")

met_deph_P2 = mean_metrics_for_param(dephasing_kraus, p_deph_P2, DEPTH, STATES)
met_depo_P2 = mean_metrics_for_param(depolarizing_kraus, p_depo_P2, DEPTH, STATES)
met_ad_P2 = mean_metrics_for_param(amplitude_damping_kraus, g_ad_P2, DEPTH, STATES)

df_P2 = pd.DataFrame([
    {"protocol": "Equal-F", "channel": "dephasing", "param": p_deph_P2, **met_deph_P2},
    {"protocol": "Equal-F", "channel": "depolarizing", "param": p_depo_P2, **met_depo_P2},
    {"protocol": "Equal-F", "channel": "amplitude_damping", "param": g_ad_P2, **met_ad_P2},
])

# Equal-F bar plot
plt.figure(figsize=(6.8, 3.8))
cats = ["dephasing", "depolarizing", "amplitude_damping"]
GDI_vals = [met_deph_P2["GDI"], met_depo_P2["GDI"], met_ad_P2["GDI"]]
GAC_vals = [met_deph_P2["GAC"], met_depo_P2["GAC"], met_ad_P2["GAC"]]

x = np.arange(len(cats))
w = 0.35

plt.bar(x - w / 2, GDI_vals, width=w, label="GDI")
plt.bar(x + w / 2, GAC_vals, width=w, label="GAC")
plt.xticks(x, ["Deph.", "Depol.", "AD"])
plt.ylabel("Index value")
plt.title("Equal final fidelity — GDI & GAC")
plt.ylim(0, 1.2)
plt.grid(alpha=0.3, axis="y")
plt.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_equalF_GDI_GAC.jpg", dpi=300, format="jpg")
plt.close()

print("Equal-F figure saved in ./results")

# -----------------------------
# Protocol P3: Equal Bures length
# -----------------------------
LB_star = ref_metrics["LB"]

p_deph_P3, _ = find_param_or_nearest(dephasing_kraus, DEPTH, STATES, LB_star, grid_deph, metric_key="LB")
p_depo_P3, _ = find_param_or_nearest(depolarizing_kraus, DEPTH, STATES, LB_star, grid_depo, metric_key="LB")
g_ad_P3, _ = find_param_or_nearest(amplitude_damping_kraus, DEPTH, STATES, LB_star, grid_ad, metric_key="LB")

met_deph_P3 = mean_metrics_for_param(dephasing_kraus, p_deph_P3, DEPTH, STATES)
met_depo_P3 = mean_metrics_for_param(depolarizing_kraus, p_depo_P3, DEPTH, STATES)
met_ad_P3 = mean_metrics_for_param(amplitude_damping_kraus, g_ad_P3, DEPTH, STATES)

df_P3 = pd.DataFrame([
    {"protocol": "Equal-LB", "channel": "dephasing", "param": p_deph_P3, **met_deph_P3},
    {"protocol": "Equal-LB", "channel": "depolarizing", "param": p_depo_P3, **met_depo_P3},
    {"protocol": "Equal-LB", "channel": "amplitude_damping", "param": g_ad_P3, **met_ad_P3},
])

# Equal-LB scatter plot
plt.figure(figsize=(5.8, 4.0))
plt.scatter(met_deph_P3["DB"], met_deph_P3["GDI"], label="Dephasing")
plt.scatter(met_depo_P3["DB"], met_depo_P3["GDI"], label="Depolarizing")
plt.scatter(met_ad_P3["DB"], met_ad_P3["GDI"], label="Amplitude damping")
plt.xlabel("Net Bures distance D_B(0,T)")
plt.ylabel("GDI (L_B / D_B)")
plt.title("Equal Bures length — geometric efficiency")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_equalLB_efficiency.jpg", dpi=300, format="jpg")
plt.close()

print("Equal-LB figure saved in ./results")

# -----------------------------
# Protocol P1: Fixed depth
# -----------------------------
p_deph_P1 = float(np.median(grid_deph))
p_depo_P1 = float(np.median(grid_depo))
g_ad_P1 = float(np.median(grid_ad))

met_deph_P1 = mean_metrics_for_param(dephasing_kraus, p_deph_P1, DEPTH, STATES)
met_depo_P1 = mean_metrics_for_param(depolarizing_kraus, p_depo_P1, DEPTH, STATES)
met_ad_P1 = mean_metrics_for_param(amplitude_damping_kraus, g_ad_P1, DEPTH, STATES)

df_P1 = pd.DataFrame([
    {"protocol": "FixedDepth", "channel": "dephasing", "param": p_deph_P1, **met_deph_P1},
    {"protocol": "FixedDepth", "channel": "depolarizing", "param": p_depo_P1, **met_depo_P1},
    {"protocol": "FixedDepth", "channel": "amplitude_damping", "param": g_ad_P1, **met_ad_P1},
])

# -----------------------------
# Save protocol summary
# -----------------------------
tab3 = pd.concat([df_P1, df_P2, df_P3], ignore_index=True)
tab3_path = f"{RESULTS_DIR}/table_protocols_summary.csv"
tab3.to_csv(tab3_path, index=False)

table_equalF_uncertainty = df_P2[df_P2["channel"].isin(["dephasing", "depolarizing"])][
    [
        "channel",
        "param",
        "F",
        "F_std",
        "DB",
        "DB_std",
        "GDI",
        "GDI_std",
        "GAC",
        "GAC_std",
        "n_valid_GDI",
        "n_valid_GAC",
        "n_total",
    ]
]

table_equalF_uncertainty_path = f"{RESULTS_DIR}/table_equalF_uncertainty.csv"
table_equalF_uncertainty.to_csv(table_equalF_uncertainty_path, index=False)

print("Protocol summary saved:", tab3_path)
print("Equal-F uncertainty table saved:", table_equalF_uncertainty_path)

print("\nEqual-F uncertainty table:")
display(table_equalF_uncertainty)

print("\nProtocol summary:")
display(tab3)

print("\nDONE.")

/tmp/ipykernel_35247/1277789750.py:88: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  sr = sqrtm(rho)
/tmp/ipykernel_35247/1277789750.py:91: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  root = sqrtm(inner)


Channel figures saved in ./results


/tmp/ipykernel_35247/1277789750.py:88: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  sr = sqrtm(rho)
/tmp/ipykernel_35247/1277789750.py:91: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  root = sqrtm(inner)


Equal-F figure saved in ./results


/tmp/ipykernel_35247/1277789750.py:88: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  sr = sqrtm(rho)
/tmp/ipykernel_35247/1277789750.py:91: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  root = sqrtm(inner)


...

Protocol summary saved: ./results/table_protocols_summary.csv
Equal-F uncertainty table saved: ./results/table_equalF_uncertainty.csv

Equal-F uncertainty table:


,channel,param,F,F_std,DB,DB_std,GDI,GDI_std,GAC,GAC_std,n_valid_GDI,n_valid_GAC,n_total
0,dephasing,0.017500,0.75180,1.012022e-01,0.49433,1.611106e-01,1.028029,1.222162e-02,0.990546,3.979449e-03,28,28,30
1,depolarizing,0.021211,0.75178,5.358537e-09,0.51565,5.992607e-09,1.011320,6.841195e-08,0.995572,8.315827e-09,30,30,30



Protocol summary:


,protocol,channel,param,F,F_std,DB,DB_std,GDI,GDI_std,GAC,GAC_std,LB,LB_std,max_aB,max_aB_std,n_valid_GDI,n_valid_GAC,n_total
0,FixedDepth,dephasing,0.017500,0.751800,1.012022e-01,0.494330,1.611106e-01,1.028029,1.222162e-02,0.990546,3.979449e-03,0.507176,1.625645e-01,0.053301,1.710345e-02,28,28,30
1,FixedDepth,depolarizing,0.045000,0.614572,5.045122e-09,0.657349,4.895077e-09,1.018692,3.717200e-08,0.994086,3.738729e-09,0.669636,2.611774e-08,0.074472,2.611727e-08,30,30,30
2,FixedDepth,amplitude_damping,0.900000,0.566159,2.556478e-01,0.687858,2.917219e-01,1.080260,3.552161e-02,0.997550,7.736073e-04,0.749866,3.303750e-01,0.322834,1.783856e-01,29,29,30
3,Equal-F,dephasing,0.017500,0.751800,1.012022e-01,0.494330,1.611106e-01,1.028029,1.222162e-02,0.990546,3.979449e-03,0.507176,1.625645e-01,0.053301,1.710345e-02,28,28,30
4,Equal-F,depolarizing,0.021211,0.751780,5.358537e-09,0.515650,5.992607e-09,1.011320,6.841195e-08,0.995572,8.315827e-09,0.521487,3.650098e-08,0.050738,3.650094e-08,30,30,30
5,Equal-F,amplitude_damping,0.843333,0.566159,2.556478e-01,0.687858,2.917219e-01,1.098634,4.179548e-02,0.996658,9.505824e-04,0.763552,3.376527e-01,0.270198,1.604718e-01,29,29,30
6,Equal-LB,dephasing,0.017500,0.751800,1.012022e-01,0.494330,1.611106e-01,1.028029,1.222162e-02,0.990546,3.979449e-03,0.507176,1.625645e-01,0.053301,1.710345e-02,28,28,30
7,Equal-LB,depolarizing,0.019746,0.764122,6.454601e-09,0.501716,7.358689e-09,1.010701,7.257918e-08,0.995762,8.925978e-09,0.507084,3.871505e-08,0.048932,3.871463e-08,30,30,30
8,Equal-LB,amplitude_damping,1.000000,0.566159,2.556478e-01,0.687858,2.917219e-01,1.000000,0.000000e+00,1.000000,0.000000e+00,0.687858,2.917219e-01,0.687858,2.917219e-01,29,29,30



DONE.
